# StreetLeague - Systeme de Recommandation d'Exercices (IA/ML)

## Projet : Coaching Intelligent

### Algorithmes utilises (Machine Learning classique, PAS un LLM) :
- **TF-IDF** (Term Frequency-Inverse Document Frequency) : Vectorisation textuelle
- **KNN** (K-Nearest Neighbors) : Recherche des exercices similaires
- **Cosine Similarity** : Mesure de distance entre vecteurs
- **K-Means Clustering** : Regroupement non-supervise
- **PCA** (Principal Component Analysis) : Reduction de dimensionnalite
- **MinMaxScaler** : Normalisation des features numeriques

### Architecture d'integration :
```
Notebook (train) -> model/*.joblib -> Flask API (app.py) -> Spring Boot -> Frontend
```

### Ce n'est PAS un LLM :
Ce systeme utilise du **Content-Based Filtering** avec des algorithmes ML classiques.
Il ne genere pas de texte — il CLASSE et RECOMMANDE des exercices existants.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy.sparse import hstack, csr_matrix
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print('=' * 60)
print('  StreetLeague - ML Exercise Recommendation System')
print('=' * 60)
print('Algorithmes : TF-IDF + KNN + Cosine Similarity')
print('Type : Content-Based Filtering')
print('Imports OK')


In [ ]:
df = pd.read_csv('data/exercises_dataset.csv')
print(f'Dataset charge : {df.shape[0]} exercices, {df.shape[1]} colonnes')
print(f'Colonnes : {list(df.columns)}')
df.head(10)


In [ ]:
print('=== Statistiques descriptives ===')
print(df.describe())
print('\n=== Distribution par type ===')
print(df['type'].value_counts())
print('\n=== Distribution par difficulte ===')
print(df['difficulte'].value_counts())
print('\n=== Valeurs manquantes ===')
print(df.isnull().sum())


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.countplot(data=df, x='type', ax=axes[0, 0], palette='viridis')
axes[0, 0].set_title('Distribution par type')
sns.countplot(data=df, x='difficulte', ax=axes[0, 1], palette='magma', order=['DEBUTANT', 'INTERMEDIAIRE', 'AVANCE'])
axes[0, 1].set_title('Distribution par difficulte')
sns.boxplot(data=df, x='type', y='dureeMinutes', ax=axes[1, 0], palette='coolwarm')
axes[1, 0].set_title('Duree par type')
sns.boxplot(data=df, x='type', y='intensite_score', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Intensite par type')
plt.tight_layout()
plt.savefig('data/eda_visualizations.png', dpi=100)
plt.show()


In [ ]:
corr = df[['dureeMinutes', 'intensite_score', 'calories_estimees']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matrice de correlation')
plt.tight_layout()
plt.savefig('data/correlation_matrix.png', dpi=100)
plt.show()


In [ ]:
# Feature Engineering : combiner texte + numerique
df['text_features'] = (df['type'] + ' ' + df['difficulte'] + ' ' + df['objectif'] + ' ' + df['muscles_cibles'] + ' ' + df['niveauRecommande'] + ' ' + df['equipement'])

le_type = LabelEncoder()
le_diff = LabelEncoder()
df['type_encoded'] = le_type.fit_transform(df['type'])
df['difficulte_encoded'] = le_diff.fit_transform(df['difficulte'])

print('Feature engineering termine')
print(f'Exemple : {df["text_features"].iloc[0][:80]}...')


In [ ]:
# TF-IDF : transforme le texte en vecteurs numeriques ponderes
tfidf = TfidfVectorizer(max_features=150, ngram_range=(1, 2), min_df=1)
tfidf_matrix = tfidf.fit_transform(df['text_features'])
print(f'Matrice TF-IDF : {tfidf_matrix.shape}')
print(f'Vocabulaire size : {len(tfidf.get_feature_names_out())}')


In [ ]:
# Combiner TF-IDF + features numeriques normalisees
scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(df[['intensite_score', 'dureeMinutes', 'calories_estimees', 'type_encoded', 'difficulte_encoded']])
combined_features = hstack([tfidf_matrix, csr_matrix(numeric_features)])
print(f'Features combinees : {combined_features.shape}')

# KNN : trouve les K voisins les plus proches (cosine similarity)
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn_model.fit(combined_features)
print(f'Modele KNN entraine : K=6, metric=cosine')


In [ ]:
def recommend(context, top_n=6):
    query_parts = [context.get('typeSeance',''), context.get('intensite',''), context.get('objectifProgramme',''), context.get('niveauJoueurs','')]
    query_text = ' '.join([p for p in query_parts if p])
    query_tfidf = tfidf.transform([query_text])
    imap = {'FAIBLE':3,'MOYENNE':5,'FORTE':8}
    tmap = {'FORCE':1,'CARDIO':0,'MOBILITE':2,'TECHNIQUE':3}
    dmap = {'DEBUTANT':1,'INTERMEDIAIRE':2,'AVANCE':0}
    iv = imap.get(context.get('intensite','MOYENNE').upper(), 5)
    tv = tmap.get(context.get('typeSeance','FORCE').upper(), 0)
    dv = dmap.get(context.get('niveauJoueurs','INTERMEDIAIRE').upper(), 1)
    dur = context.get('dureeSeanceMinutes', 60) / 60.0 * 20
    cal = iv * 20
    qn = scaler.transform([[iv, dur, cal, tv, dv]])
    qc = hstack([query_tfidf, csr_matrix(qn)])
    distances, indices = knn_model.kneighbors(qc, n_neighbors=top_n)
    results = df.iloc[indices[0]].copy()
    results['score'] = ((1 - distances[0]) * 100).round(1)
    return results[['nom','type','difficulte','dureeMinutes','equipement','objectif','score']].reset_index(drop=True)

# Test
print('TEST : Seance CARDIO intense')
print(recommend({'typeSeance':'CARDIO','intensite':'FORTE','objectifProgramme':'endurance vitesse','niveauJoueurs':'AVANCE'}).to_string())


In [ ]:
# Evaluation : Precision@6 par type
types = ['FORCE', 'CARDIO', 'MOBILITE', 'TECHNIQUE']
scores = []
for t in types:
    r = recommend({'typeSeance': t, 'intensite': 'MOYENNE', 'objectifProgramme': t.lower(), 'niveauJoueurs': 'INTERMEDIAIRE'})
    correct = (r['type'] == t).sum()
    p = correct / len(r)
    scores.append(p)
    print(f'{t:12s} : Precision@6 = {p:.0%} ({correct}/6)')
avg = np.mean(scores)
print(f'\nMoyenne Precision@6 : {avg:.0%}')


In [ ]:
# PCA + K-Means clustering
pca = PCA(n_components=2)
features_2d = pca.fit_transform(combined_features.toarray())
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(combined_features.toarray())

plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=df['type_encoded'], cmap='viridis', s=60, alpha=0.7)
plt.colorbar(scatter, label='Type')
plt.title('PCA - Exercices projetes en 2D (couleur = type)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.savefig('data/pca_clusters.png', dpi=100)
plt.show()
print(f'Variance expliquee : {pca.explained_variance_ratio_.sum():.1%}')


In [ ]:
# Export du modele pour le service Flask
os.makedirs('model', exist_ok=True)
joblib.dump(knn_model, 'model/knn_model.joblib')
joblib.dump(tfidf, 'model/tfidf_vectorizer.joblib')
joblib.dump(scaler, 'model/scaler.joblib')
joblib.dump(le_type, 'model/label_encoder_type.joblib')
joblib.dump(le_diff, 'model/label_encoder_difficulte.joblib')
joblib.dump(combined_features, 'model/combined_features.joblib')
df.to_csv('model/exercises_enriched.csv', index=False)

print('Export termine !')
for f in sorted(os.listdir('model')):
    print(f'  {f} ({os.path.getsize("model/" + f) / 1024:.1f} KB)')
print('\nLancer app.py pour demarrer le service de recommandation.')
